# 06 Classification: XGBoost trained on synthetic data, evaluated on real test data

**Purpose:** measure predictive utility (Objective 4 of the proposal). Does a classifier trained only on synthetic student records predict engagement for real, unseen students?

**Data rules**
- Classifier features: only `nevents, ndays_act, ncontent, nforum_posts`, in this fixed order. `Engagement_Level` is used only as the training target. `completed_%` does not exist in any of these files.
- Synthetic training rows keep their NaNs, because XGBoost handles missing values natively.
- **The real test set (`test.csv`, 5,333 rows) is used only for the final metrics.** Its class distribution (about 82.5% class 0) is deliberately left imbalanced, because it must reflect real data.
- **Leakage controls:** no hyperparameter, threshold or model selection is done on the test set. XGBoost parameters are fixed in advance (`n_estimators=200, max_depth=4, learning_rate=0.1`, seed 42) and identical for every model.

**Experiments**

| Section | Strategy | Training data | Models |
|---|---|---|---|
| 1 | `random` | unbalanced synthetic (target variant) | real-train reference, CTGAN, TVAE |
| 2 | `random` | balanced synthetic (conditional sampling) | CTGAN, TVAE |
| 3 | `from_column` | unbalanced synthetic (target variant) | CTGAN, TVAE |
| 4 | all | side-by-side comparison | in memory |

The **real-train reference** is trained on real `train.csv`. It is a ceiling for comparison and is not part of the paper's method.

**Metrics:** weighted precision, recall and F1 (as in the paper), per-class metrics, and the confusion matrix. Weighted scores are dominated by class 0, so **class-1 metrics and error counts are read alongside them**.

**Outputs:** `reports/xgb_random_target.json`, `reports/xgb_random_target_balanced.json`, `reports/xgb_from_column_target.json`, `reports/classification_summary.json`, and models in `artifacts/models/` (reused by `07_explainability.ipynb`).

**Paper values (reference only, not targets):** unbalanced F1 with random missingness CTGAN 85%, TVAE 90%. With from-column CTGAN 88%, TVAE 89%. Balanced F1 CTGAN 84%, TVAE 86% (random), CTGAN 83%, TVAE 85% (from-column).

**Reproducibility check:** the weighted F1 values from earlier runs are listed in the setup cell, and each section prints whether the recomputed values match them.

In [1]:
import json
from datetime import datetime, timezone
from pathlib import Path

import joblib
import pandas as pd
import sklearn
import xgboost
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support
from xgboost import XGBClassifier

# ---------- Settings (to move into configs/base.yaml) ----------
FEATURES = ["nevents", "ndays_act", "ncontent", "nforum_posts"]
TARGET = "Engagement_Level"
SEED = 42
XGB_PARAMS = dict(
    n_estimators=200, max_depth=4, learning_rate=0.1,
    random_state=SEED, eval_metric="logloss",
)

# Weighted F1 recorded in earlier runs (reproducibility check). from_column has none yet.
EXPECTED_W_F1 = {
    "real_train_reference": 0.9065,
    "ctgan_random_target": 0.8669,
    "tvae_random_target": 0.8956,
    "ctgan_random_target_balanced": 0.8043,
    "tvae_random_target_balanced": 0.8300,
}


# ---------- Helpers ----------
def find_project_root(start=None):
    """Walk upwards until a folder containing configs/base.yaml is found."""
    start = Path(start or Path.cwd()).resolve()
    for folder in [start, *start.parents]:
        if (folder / "configs" / "base.yaml").exists():
            return folder
    raise FileNotFoundError(f"Could not find configs/base.yaml above {start}")


def check_columns(df: pd.DataFrame, label: str) -> pd.DataFrame:
    """Return df restricted to features + target, in the fixed order."""
    expected = FEATURES + [TARGET]
    missing = [c for c in expected if c not in df.columns]
    if missing:
        raise ValueError(f"{label}: missing columns {missing}")
    return df[expected]


def evaluate(model: XGBClassifier, test: pd.DataFrame) -> dict:
    """Weighted and per-class metrics on the real test set."""
    y_true = test[TARGET].astype(int)
    y_pred = model.predict(test[FEATURES])
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )
    per_class = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    return {
        "weighted": {"precision": float(p), "recall": float(r), "f1": float(f1)},
        "per_class": {k: per_class[k] for k in ("0", "1")},
        "confusion_matrix": confusion_matrix(y_true, y_pred).tolist(),
        "test_class_counts": {str(k): int(v) for k, v in y_true.value_counts().sort_index().items()},
    }


def run_experiment(name: str, train_df: pd.DataFrame) -> dict:
    """Train on train_df (features + label), save the model, evaluate on the real test set."""
    train_df = check_columns(train_df, name)
    model = XGBClassifier(**XGB_PARAMS)
    model.fit(train_df[FEATURES], train_df[TARGET].astype(int))
    joblib.dump(model, MODEL_DIR / f"{name}.joblib")

    result = evaluate(model, TEST)
    result["train_rows"] = len(train_df)
    result["train_class_share"] = {
        str(k): round(float(v), 4)
        for k, v in train_df[TARGET].value_counts(normalize=True).sort_index().items()
    }
    result["train_feature_nan_pct"] = (train_df[FEATURES].isna().mean() * 100).round(2).to_dict()
    return result


def summarise(results: dict) -> pd.DataFrame:
    """One column per model: weighted and class-1 metrics plus error counts."""
    return pd.DataFrame({
        name: {
            "w_precision": r["weighted"]["precision"],
            "w_recall": r["weighted"]["recall"],
            "w_f1": r["weighted"]["f1"],
            "class1_precision": r["per_class"]["1"]["precision"],
            "class1_recall": r["per_class"]["1"]["recall"],
            "class1_f1": r["per_class"]["1"]["f1-score"],
            "false_pos": r["confusion_matrix"][0][1],
            "false_neg": r["confusion_matrix"][1][0],
        }
        for name, r in results.items()
    }).round(4)


def run_group(key: str, strategy: str, variant: str, sources: dict) -> dict:
    """Run every experiment in a group, print results, check reproducibility, save JSON."""
    results = {name: run_experiment(name, df) for name, df in sources.items()}

    print(summarise(results).to_string())
    for name, r in results.items():
        print(f"\n{name}")
        print("  training NaN % per feature:", r["train_feature_nan_pct"])
        print("  confusion matrix [[TN, FP], [FN, TP]]:", r["confusion_matrix"])
        expected = EXPECTED_W_F1.get(name)
        if expected is not None:
            ok = abs(r["weighted"]["f1"] - expected) < 5e-4
            print(f"  reproducibility vs earlier run (weighted F1 {expected}):",
                  "match" if ok else f"DIFFERS (now {r['weighted']['f1']:.4f})")

    payload = {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "seed": SEED,
        "xgb_params": XGB_PARAMS,
        "features": FEATURES,
        "target": TARGET,
        "strategy": strategy,
        "variant": variant,
        "evaluated_on": "real test set (final evaluation only; no tuning)",
        "versions": {"xgboost": xgboost.__version__, "sklearn": sklearn.__version__, "pandas": pd.__version__},
        "experiments": results,
    }
    out = REPORT_DIR / f"xgb_{key}.json"
    out.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print("\nSaved:", out.name)
    return results


# ---------- Paths and shared inputs ----------
PROJECT_ROOT = find_project_root()
PROC = PROJECT_ROOT / "data" / "processed"
GEN_DIR = PROJECT_ROOT / "artifacts" / "generators"
MODEL_DIR = PROJECT_ROOT / "artifacts" / "models"
REPORT_DIR = PROJECT_ROOT / "reports"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(exist_ok=True)

# Real test set: loaded once, used for final evaluation only
TEST = check_columns(pd.read_csv(PROC / "test.csv"), "test")
all_results = {}

print("xgboost:", xgboost.__version__, "| sklearn:", sklearn.__version__)
print("Project root:", PROJECT_ROOT)
print("Test:", TEST.shape, "| class share:", TEST[TARGET].value_counts(normalize=True).round(4).to_dict())
print("Test NaN % per feature:", (TEST[FEATURES].isna().mean() * 100).round(2).to_dict())

xgboost: 3.2.0 | sklearn: 1.9.1
Project root: D:\student_engagement_twin
Test: (5333, 5) | class share: {0: 0.8247, 1: 0.1753}
Test NaN % per feature: {'nevents': 17.1, 'ndays_act': 10.09, 'ncontent': 0.0, 'nforum_posts': 67.84}


## 1. `random` strategy, unbalanced synthetic training data

**Goal:** the paper's main experiment. Train XGBoost on synthetic data from CTGAN and TVAE (target as generator column, NaNs restored at the real training rates) and evaluate on the real test set. A model trained on real `train.csv` is included as a reference ceiling.

**Training files:** `ctgan_random_target_restored.csv`, `tvae_random_target_restored.csv`, and real `train.csv`.

**Expected (earlier run):** weighted F1 0.9065 (real reference), 0.8669 (CTGAN), 0.8956 (TVAE). The real-data reference should be the best or close to it. A synthetic model beating it by a wide margin would be surprising and would need an explanation.

**Output:** `reports/xgb_random_target.json`.

In [2]:
sources = {
    "real_train_reference": pd.read_csv(PROC / "train.csv"),
    "ctgan_random_target": pd.read_csv(GEN_DIR / "ctgan_random_target_restored.csv"),
    "tvae_random_target": pd.read_csv(GEN_DIR / "tvae_random_target_restored.csv"),
}
all_results["random_unbalanced"] = run_group(
    key="random_target", strategy="random",
    variant="target as generator column (unbalanced)", sources=sources,
)

                  real_train_reference  ctgan_random_target  tvae_random_target
w_precision                     0.9055               0.8665              0.8944
w_recall                        0.9087               0.8766              0.8995
w_f1                            0.9065               0.8669              0.8956
class1_precision                0.7692               0.7174              0.7588
class1_recall                   0.6845               0.4888              0.6257
class1_f1                       0.7244               0.5814              0.6858
false_pos                     192.0000             180.0000            186.0000
false_neg                     295.0000             478.0000            350.0000

real_train_reference
  training NaN % per feature: {'nevents': 17.08, 'ndays_act': 10.53, 'ncontent': 0.0, 'nforum_posts': 68.11}
  confusion matrix [[TN, FP], [FN, TP]]: [[4206, 192], [295, 640]]
  reproducibility vs earlier run (weighted F1 0.9065): match

ctgan_random_target


## 2. `random` strategy, balanced synthetic training data

**Goal:** reproduce the paper's balanced experiment. The training sets contain 10,664 rows per class, created by conditional sampling from the saved generators.

**Training files:** `ctgan_random_target_balanced_restored.csv`, `tvae_random_target_balanced_restored.csv`.

**Evaluation:** the same real test set, still about 82.5% class 0. The test distribution is **not** balanced.

**Hypothesis to test (not a finding):** training on 50/50 data shifts the model towards predicting class 1. Class-1 recall should rise and precision should fall. Weighted scores can drop even if the model finds more class-1 students, because weighted metrics are dominated by class 0.

**Expected (earlier run):** weighted F1 0.8043 (CTGAN) and 0.8300 (TVAE).

**Output:** `reports/xgb_random_target_balanced.json`.

In [3]:
sources = {
    "ctgan_random_target_balanced": pd.read_csv(GEN_DIR / "ctgan_random_target_balanced_restored.csv"),
    "tvae_random_target_balanced": pd.read_csv(GEN_DIR / "tvae_random_target_balanced_restored.csv"),
}
for name, df in sources.items():
    counts = df[TARGET].value_counts().sort_index().to_dict()
    assert counts == {0: 10664, 1: 10664}, f"{name}: training set is not balanced: {counts}"

all_results["random_balanced"] = run_group(
    key="random_target_balanced", strategy="random",
    variant="target as generator column, balanced by conditional sampling", sources=sources,
)

                  ctgan_random_target_balanced  tvae_random_target_balanced
w_precision                             0.8606                       0.8874
w_recall                                0.7825                       0.8108
w_f1                                    0.8043                       0.8300
class1_precision                        0.4355                       0.4790
class1_recall                           0.8118                       0.9016
class1_f1                               0.5668                       0.6256
false_pos                             984.0000                     917.0000
false_neg                             176.0000                      92.0000

ctgan_random_target_balanced
  training NaN % per feature: {'nevents': 17.12, 'ndays_act': 10.48, 'ncontent': 0.0, 'nforum_posts': 68.48}
  confusion matrix [[TN, FP], [FN, TP]]: [[3414, 984], [176, 759]]
  reproducibility vs earlier run (weighted F1 0.8043): match

tvae_random_target_balanced
  training NaN % per

## 3. `from_column` strategy, unbalanced synthetic training data

**Goal:** the missingness test of the project. The generators modelled the missingness flags, so the restored NaNs can keep the link between missing values and engagement. Does that change how well a classifier trained on synthetic data predicts engagement on real students?

**Training files:** `ctgan_from_column_target_restored.csv`, `tvae_from_column_target_restored.csv`.

**Hypotheses to test (not findings)**
- Because `from_column` keeps the missingness-engagement link, class-1 recall could improve compared with `random`.
- TVAE's restored data has almost no NaNs in `ndays_act` (flag rate 0.0037 against 0.1053 in real train), so TVAE may handle real test rows with a missing `ndays_act` less well. The printed training NaN percentages show this.

**Compare with:** section 1 (same unbalanced setting, different missingness strategy).

**No earlier run exists** for this section, so there is no reproducibility check here.

**Output:** `reports/xgb_from_column_target.json`.

In [4]:
sources = {
    "ctgan_from_column_target": pd.read_csv(GEN_DIR / "ctgan_from_column_target_restored.csv"),
    "tvae_from_column_target": pd.read_csv(GEN_DIR / "tvae_from_column_target_restored.csv"),
}
all_results["from_column_unbalanced"] = run_group(
    key="from_column_target", strategy="from_column",
    variant="target as generator column (unbalanced)", sources=sources,
)

                  ctgan_from_column_target  tvae_from_column_target
w_precision                         0.8936                   0.8972
w_recall                            0.8987                   0.8952
w_f1                                0.8948                   0.8961
class1_precision                    0.7562                   0.6922
class1_recall                       0.6235                   0.7241
class1_f1                           0.6835                   0.7078
false_pos                         188.0000                 301.0000
false_neg                         352.0000                 258.0000

ctgan_from_column_target
  training NaN % per feature: {'nevents': 25.79, 'ndays_act': 6.64, 'ncontent': 0.0, 'nforum_posts': 70.83}
  confusion matrix [[TN, FP], [FN, TP]]: [[4210, 188], [352, 583]]

tvae_from_column_target
  training NaN % per feature: {'nevents': 15.09, 'ndays_act': 0.37, 'ncontent': 0.0, 'nforum_posts': 64.62}
  confusion matrix [[TN, FP], [FN, TP]]: [[4097, 301],

## 4. Side-by-side comparison and summary

**Goal:** one table with every model, and a saved summary for the report and the dashboard.

**Columns**
- `real_train_reference`: ceiling, trained on real data.
- `<generator>_random`: section 1. `<generator>_random_balanced`: section 2. `<generator>_from_column`: section 3.

**Rows:** weighted precision, recall and F1; class-1 precision, recall and F1; false positives (low-engagement students flagged as high); false negatives (high-engagement students missed).

**Reading the table**
- The error counts show the real trade-off. Weighted F1 hides it, because about 82.5% of the test rows are class 0.
- Differences between generators and strategies come from **one seed**. They are read as observations, and no winner is claimed from small gaps.
- The test set has genuine NaNs (about 17% / 10% / 0% / 68%). Models trained on data with a different NaN pattern may behave differently on them.

**Interpretation limits (for the report):** predictions are exploratory decision support, not grounds for decisions about individual students. Missing forum activity is not proof of low engagement.

**Output:** `reports/classification_summary.json`.

In [5]:
combined = {
    "real_train_reference": all_results["random_unbalanced"]["real_train_reference"],
    "ctgan_random": all_results["random_unbalanced"]["ctgan_random_target"],
    "ctgan_random_balanced": all_results["random_balanced"]["ctgan_random_target_balanced"],
    "ctgan_from_column": all_results["from_column_unbalanced"]["ctgan_from_column_target"],
    "tvae_random": all_results["random_unbalanced"]["tvae_random_target"],
    "tvae_random_balanced": all_results["random_balanced"]["tvae_random_target_balanced"],
    "tvae_from_column": all_results["from_column_unbalanced"]["tvae_from_column_target"],
}

summary = summarise(combined)
print(summary.to_string())

payload = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "evaluated_on": "real test set (5,333 rows); final evaluation only",
    "xgb_params": XGB_PARAMS,
    "seed": SEED,
    "versions": {"xgboost": xgboost.__version__, "sklearn": sklearn.__version__, "pandas": pd.__version__},
    "summary": summary.to_dict(),
    "models_saved_in": "artifacts/models/",
}
out = REPORT_DIR / "classification_summary.json"
out.write_text(json.dumps(payload, indent=2), encoding="utf-8")
print("\nSaved:", out)

                  real_train_reference  ctgan_random  ctgan_random_balanced  ctgan_from_column  tvae_random  tvae_random_balanced  tvae_from_column
w_precision                     0.9055        0.8665                 0.8606             0.8936       0.8944                0.8874            0.8972
w_recall                        0.9087        0.8766                 0.7825             0.8987       0.8995                0.8108            0.8952
w_f1                            0.9065        0.8669                 0.8043             0.8948       0.8956                0.8300            0.8961
class1_precision                0.7692        0.7174                 0.4355             0.7562       0.7588                0.4790            0.6922
class1_recall                   0.6845        0.4888                 0.8118             0.6235       0.6257                0.9016            0.7241
class1_f1                       0.7244        0.5814                 0.5668             0.6835       0.6858     